In [1]:
import pandas as pd
from scipy import stats
import sys
sys.path.append('../src')
from enrich import enrich_listings

df = enrich_listings()
df_valid = df[(df['price_outlier_flag']==False) & (df['price_clean'].notna())]

entire = df_valid[df_valid['room_type']=='Entire home/apt']['price_clean']
private = df_valid[df_valid['room_type']=='Private room']['price_clean']

print(f"Entire home: n={len(entire)}, mean={entire.mean():.2f}, std={entire.std():.2f}")
print(f"Private room: n={len(private)}, mean={private.mean():.2f}, std={private.std():.2f}")

# Check normality assumption first
print(stats.shapiro(entire.sample(min(5000, len(entire)), random_state=42)))
print(stats.shapiro(private.sample(min(5000, len(private)), random_state=42)))

Loaded listings: 10480 rows, 79 columns
Price cleaned. Outliers flagged: 7

Date parsing check:
   host_since host_since_parsed
0  2010-03-23        2010-03-23
1  2010-05-13        2010-05-13
2  2010-05-13        2010-05-13
3  2010-08-08        2010-08-08
4  2010-09-01        2010-09-01

Unparseable dates per column:
  last_scraped: 0 unparseable (out of 10480 non-null originals)
  first_review: 1097 unparseable (out of 9383 non-null originals)
  last_review: 1097 unparseable (out of 9383 non-null originals)
  host_since: 3 unparseable (out of 10477 non-null originals)

Property type grouping:
property_type_grouped
Entire place      8214
Private room      1371
Hotel room         418
Houseboat/Boat     392
Other               30
Shared room         29
Room in other       26
Name: count, dtype: int64

Columns after dropping unreliable ones: 85
Missing flags: beds=4576, bathrooms=4548, price=4606

Coordinate precision after standardization:
latitude
5    9349
4    1019
3     105
2       7

In [2]:
u_stat, p_value = stats.mannwhitneyu(entire, private, alternative='two-sided')
print(f"U statistic: {u_stat}")
print(f"p-value: {p_value}")

# Effect size: rank-biserial correlation (appropriate for Mann-Whitney)
n1, n2 = len(entire), len(private)
r = 1 - (2*u_stat) / (n1 * n2)
print(f"Rank-biserial correlation (effect size): {r:.4f}")

U statistic: 4680826.5
p-value: 4.953350071257756e-253
Rank-biserial correlation (effect size): -0.6221


In [ ]:
##-----------------H2 : Superhost vs. Non-Superhost Review Scores-------------------------
superhost_scores = df[df['host_is_superhost']=='t']['review_scores_rating'].dropna()
non_superhost_scores = df[df['host_is_superhost']=='f']['review_scores_rating'].dropna()

print(stats.shapiro(superhost_scores.sample(min(5000, len(superhost_scores)), random_state=42)))
print(stats.shapiro(non_superhost_scores.sample(min(5000, len(non_superhost_scores)), random_state=42)))
print(f"Superhost: n={len(superhost_scores)}, mean={superhost_scores.mean():.3f}")
print(f"Non-superhost: n={len(non_superhost_scores)}, mean={non_superhost_scores.mean():.3f}")

ShapiroResult(statistic=np.float64(0.6172392769729891), pvalue=np.float64(5.192255589434924e-53))
ShapiroResult(statistic=np.float64(0.5932040804608053), pvalue=np.float64(9.427460660625239e-76))
Superhost: n=1813, mean=4.865
Non-superhost: n=7460, mean=4.839


In [4]:
u_stat2, p_value2 = stats.mannwhitneyu(superhost_scores, non_superhost_scores, alternative='two-sided')
n1, n2 = len(superhost_scores), len(non_superhost_scores)
r2 = 1 - (2*u_stat2) / (n1 * n2)

print(f"U statistic: {u_stat2}")
print(f"p-value: {p_value2}")
print(f"Rank-biserial correlation (effect size): {r2:.4f}")

U statistic: 5894053.0
p-value: 2.0135998888920313e-18
Rank-biserial correlation (effect size): 0.1284


In [ ]:
##----H3 : Listings with >10 reviews vs. ≤10 reviews, price difference--------
high_reviews = df_valid[df_valid['review_count_computed'] > 10]['price_clean']
low_reviews = df_valid[df_valid['review_count_computed'] <= 10]['price_clean']

print(stats.shapiro(high_reviews.sample(min(5000, len(high_reviews)), random_state=42)))
print(stats.shapiro(low_reviews.sample(min(5000, len(low_reviews)), random_state=42)))
print(f">10 reviews: n={len(high_reviews)}, mean={high_reviews.mean():.2f}")
print(f"<=10 reviews: n={len(low_reviews)}, mean={low_reviews.mean():.2f}")

ShapiroResult(statistic=np.float64(0.2970516342034646), pvalue=np.float64(1.4283564879305937e-75))
ShapiroResult(statistic=np.float64(0.21552478565566846), pvalue=np.float64(6.893019874040519e-76))
>10 reviews: n=3054, mean=257.32
<=10 reviews: n=2813, mean=288.79


In [6]:
u_stat3, p_value3 = stats.mannwhitneyu(high_reviews, low_reviews, alternative='two-sided')
n1, n2 = len(high_reviews), len(low_reviews)
r3 = 1 - (2*u_stat3) / (n1 * n2)

print(f"U statistic: {u_stat3}")
print(f"p-value: {p_value3}")
print(f"Rank-biserial correlation (effect size): {r3:.4f}")

U statistic: 3627805.5
p-value: 6.975502511547606e-25
Rank-biserial correlation (effect size): 0.1554


In [ ]:
##---H4 : Neighbourhood Price Differences (ANOVA)----------------
neighbourhood_groups = [group['price_clean'].dropna().values 
                          for name, group in df_valid.groupby('neighbourhood_cleansed')]

# Check homogeneity of variance (Levene's test) - an ANOVA assumption
levene_stat, levene_p = stats.levene(*neighbourhood_groups)
print(f"Levene's test: statistic={levene_stat:.4f}, p-value={levene_p}")

Levene's test: statistic=3.4195, p-value=2.025219661302865e-07


In [8]:
h_stat, p_value4 = stats.kruskal(*neighbourhood_groups)
print(f"H statistic: {h_stat:.4f}")
print(f"p-value: {p_value4}")

# Effect size for Kruskal-Wallis: eta-squared approximation
n_total = sum(len(g) for g in neighbourhood_groups)
k_groups = len(neighbourhood_groups)
eta_squared = (h_stat - k_groups + 1) / (n_total - k_groups)
print(f"Eta-squared (effect size): {eta_squared:.4f}")

H statistic: 417.2274
p-value: 2.5105639011629185e-75
Eta-squared (effect size): 0.0678
